# 03 · Experimentos (Colab T4)

Corre el diseño factorial completo (LoRA y DoRA) con reanudación. **Forzar GPU T4** (Entorno de ejecución → Cambiar tipo de entorno → T4) para que todas las corridas sean comparables. Cada corrida guarda su resultado apenas termina; si Colab se desconecta, vuelve a ejecutar esta celda y continúa donde quedó.

In [ ]:
# === Setup en Google Colab (ejecutar solo en Colab) ===
# Descomenta estas líneas la primera vez en Colab.
# Rama con los splits que incluyen los datos manuales (cambiar a main cuando se fusione):
# !git clone -b splits-con-manual https://github.com/Hammonne/triaje-clinico-lora.git
# %cd triaje-clinico-lora
# Colab trae torchao 0.10, que peft 0.21 rechaza (pide >= 0.16) y no usamos:
# !pip uninstall -q -y torchao
# Colab ya trae todas las dependencias; estas líneas solo verifican mínimos e instalan el paquete:
# !pip install -q -r requirements.txt
# !pip install -q -e .
# from google.colab import drive; drive.mount('/content/drive')

In [ ]:
# === Celda de configuración (única fuente de parámetros) ===
import sys; sys.path.insert(0, "src")
from pathlib import Path
from triaje.config import Config, set_seed

config = Config()
# En Colab, apuntar las salidas a Drive para sobrevivir desconexiones:
# config.drive_dir = Path("/content/drive/MyDrive/triaje")
# config.results_dir = config.drive_dir / "results"
# config.output_dir  = config.drive_dir / "adaptadores"
set_seed(config.seed)
print("run_id ejemplo:", config.run_id(), "| modelo:", config.model_name)

## Verificar GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No hay GPU. Activa la T4 en Colab."
print("GPU:", torch.cuda.get_device_name(0))

## (Opcional) Smoke-test en CPU antes de gastar GPU

Verifica que el pipeline corre con un modelo diminuto y pocos ejemplos.

In [ ]:
# from triaje.config import Config
# from triaje.experimentos import correr_suite
# smoke = Config().aplicar_smoke_test()
# correr_suite(smoke, rs_piloto=(), rs_factorial=(4,), ratios=(1,), semillas=(42,), incluir_piloto=False)

## Correr la suite completa

23 corridas, en este orden: baseline; ronda 1 (semilla 42) con los extremos r ∈ {2, 128} y el factorial r ∈ {4, 16, 64} × α/r ∈ {1, 2}; ronda 2 (semilla 123) con lo mismo; y al final DoRA (α/r=2, r ∈ {4, 16, 64}) con ambas semillas. Revisa el tiempo de las primeras corridas para estimar el total.

In [ ]:
from triaje.experimentos import correr_suite
correr_suite(config, incluir_baseline=True)   # reanuda automáticamente